- 001 hello world

In [ ]:
from ppp_paths import databulk, dataraw, dataint


In [ ]:
import pandas as pd
import unicodedata
import re
import gc

In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')

In [ ]:
possiblepairs_papersonly = possiblepairs['magid'].drop_duplicates().reset_index(drop=True)
possiblepairs_papersonly
paperfields = pd.read_csv(databulk + "magfield_oecd_wos_crosswalk.zip", sep='\t', compression='zip', dtype=str)
paperfields = paperfields.rename(columns={'paperid': 'magid'})
paperfields['magid'] = paperfields['magid'].astype(str)
paperfields = paperfields[['magid', 'wosfield']]
paperfields = paperfields.dropna()
paperfields['wosfield'] = paperfields['wosfield'].str.replace('&amp;', '&')
paperfields = pd.merge(paperfields, possiblepairs_papersonly, on='magid', how='inner')
del possiblepairs_papersonly 
gc.collect()
 

In [ ]:
woscpcclassxwalksimple = pd.read_stata(dataraw + 'woscpcclass.dta')
woscpcclassxwalksimple = woscpcclassxwalksimple.rename(columns={'wos_namefull':'wosfield'})
woscpcsubclassxwalksimple = pd.read_stata(dataraw + 'woscpcsubclass.dta')
woscpcsubclassxwalksimple = woscpcsubclassxwalksimple.rename(columns={'wos_namefull':'wosfield'})
woscpcsubclassxwalksimple

In [ ]:
woscpcclassxwalk = pd.read_stata(dataraw + 'woscpcclassscore.dta')
woscpcclassxwalk = woscpcclassxwalk.rename(columns={'wos_namefull':'wosfield'})
woscpcclassxwalk
woscpcsubclassxwalk = pd.read_stata(dataraw + 'woscpcsubclassscore.dta')
woscpcsubclassxwalk = woscpcsubclassxwalk.rename(columns={'wos_namefull':'wosfield'})
woscpcsubclassxwalk

In [ ]:
possiblepairs_patentsonly = possiblepairs['patent_id'].drop_duplicates().reset_index(drop=True)
possiblepairs_patentsonly
patfields = pd.read_csv(databulk + 'g_cpc_at_issue.tsv.zip', sep='\t', compression='zip')
patfields = pd.merge(patfields, possiblepairs_patentsonly, on='patent_id', how='inner')
del possiblepairs_patentsonly 
gc.collect()
# patfields

In [ ]:
patfields = patfields[['patent_id', 'cpc_class', 'cpc_sequence', 'cpc_section', 'cpc_subclass']]
patfields = patfields[(patfields['cpc_sequence']<11)]
patfields.value_counts('cpc_sequence', ascending=True)
# patfields = patfields[['patent_id', 'cpc_class', 'cpc_subclass']]
patfields = patfields[['patent_id', 'cpc_subclass']]
patfields = patfields.drop_duplicates()
patfields

In [ ]:
fieldoverlap = possiblepairs[['magid', 'patent_id']]
fieldoverlap = pd.merge(fieldoverlap, patfields, on='patent_id', how='left')
fieldoverlap = pd.merge(fieldoverlap, paperfields, on='magid', how='left')
fieldoverlap = fieldoverlap.drop_duplicates()
fieldoverlap

In [ ]:
fieldoverlap = pd.merge(fieldoverlap, woscpcsubclassxwalk, on=['cpc_subclass', 'wosfield'], how='left')
fieldoverlap = fieldoverlap[['magid', 'wosfield', 'patent_id', 'cpc_subclass', 'weight_cpc4_to_wos']]
fieldoverlap

In [ ]:
fieldoverlap['weight_cpc4_to_wos'] = fieldoverlap['weight_cpc4_to_wos'].fillna(0)
fieldoverlap = fieldoverlap[['magid', 'patent_id', 'weight_cpc4_to_wos']]
fieldoverlap = fieldoverlap.groupby(['magid', 'patent_id'])['weight_cpc4_to_wos'].max().reset_index()
# fieldoverlap = fieldoverlap.groupby(['magid', 'patent_id'])['weight_cpc3_to_wos'].max().reset_index()
fieldoverlap

In [ ]:
# fieldoverlap = pd.read_parquet(dataint + 'possiblepairs_fieldoverlap.parquet')


In [ ]:
fieldoverlap = fieldoverlap.rename(columns={'weight_cpc4_to_wos': 'fieldoverlap'})

In [ ]:
fieldoverlap

In [ ]:
fieldoverlap.to_parquet(dataint + 'possiblepairs_fieldoverlap.parquet', compression='snappy')